# Understanding the Interface class

Install the Transformers, Datasets, and Evaluate libraries to run this notebook.

In [1]:
!pip install -q datasets evaluate transformers[sentencepiece] gradio torch torchaudio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.3 MB/s eta 0:00:00


In [2]:
print("=" * 80)
print("Checking NVIDIA GPU")
print("=" * 80)

!nvidia-smi

Checking NVIDIA GPU
Mon May 18 03:33:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+---------------------------

In [3]:
import os
import time
import logging

import numpy as np
import torch
import gradio as gr
from transformers import pipeline
from transformers.utils import logging as hf_logging

os.environ["GRADIO_DEBUG"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "info"

logging.basicConfig(level=logging.INFO)
hf_logging.set_verbosity_info()
logging.getLogger("gradio").setLevel(logging.INFO)

print("=" * 80)
print("Libraries imported successfully.")
print("Gradio version:", gr.__version__)
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

device = 0 if torch.cuda.is_available() else -1

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("No GPU found. Running on CPU.")

print("Using device:", "GPU" if device == 0 else "CPU")
print("=" * 80)

Libraries imported successfully.
Gradio version: 5.50.0
Torch version: 2.10.0+cu128
CUDA available: True
GPU name: Tesla T4
Using device: GPU


In [4]:
def reverse_audio(audio):
    print("=" * 80)
    print("Reverse audio function called.")

    if audio is None:
        print("No audio received.")
        print("=" * 80)
        return None

    sr, data = audio

    print("Sample rate:", sr)
    print("Audio data type:", type(data))
    print("Audio shape:", data.shape)
    print("Audio dtype:", data.dtype)

    reversed_data = np.flipud(data)

    print("Audio reversed successfully.")
    print("Output shape:", reversed_data.shape)
    print("=" * 80)

    return (sr, reversed_data)


mic = gr.Audio(
    sources=["microphone"],
    type="numpy",
    label="Speak here..."
)

reverse_demo = gr.Interface(
    fn=reverse_audio,
    inputs=mic,
    outputs=gr.Audio(label="Reversed Audio"),
    title="Reverse Microphone Audio",
    description="Record your voice, submit it, and hear the reversed audio."
)

print("Launching reverse audio demo...")
reverse_demo.launch(
    share=True,
    debug=True,
    show_error=True
)

Launching reverse audio demo...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://d7369f758fa26b7ccc.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Reverse audio function called.
No audio received.
Reverse audio function called.
Sample rate: 44100
Audio data type: <class 'numpy.ndarray'>
Audio shape: (595350,)
Audio dtype: int16
Audio reversed successfully.
Output shape: (595350,)
Created dataset file at: .gradio/flagged/dataset1.csv
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://d7369f758fa26b7ccc.gradio.live


In [5]:
notes = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]


def generate_tone(note, octave, duration):
    print("=" * 80)
    print("Generate tone function called.")
    print("Selected note index:", note)
    print("Selected note name:", notes[note])
    print("Selected octave:", octave)
    print("Selected duration:", duration)

    sr = 48000

    octave = int(octave)
    duration = float(duration)

    if duration <= 0:
        print("Invalid duration.")
        print("=" * 80)
        return None

    a4_freq = 440
    tones_from_a4 = 12 * (octave - 4) + (note - 9)
    frequency = a4_freq * 2 ** (tones_from_a4 / 12)

    print("Calculated frequency:", round(frequency, 2), "Hz")

    audio_time = np.linspace(
        0,
        duration,
        int(duration * sr),
        endpoint=False
    )

    audio = (0.5 * np.sin(audio_time * (2 * np.pi * frequency))).astype(np.float32)

    print("Generated audio sample rate:", sr)
    print("Generated audio shape:", audio.shape)
    print("Generated audio dtype:", audio.dtype)
    print("=" * 80)

    return (sr, audio)


tone_demo = gr.Interface(
    fn=generate_tone,
    inputs=[
        gr.Dropdown(
            choices=notes,
            type="index",
            label="Note",
            value="C"
        ),
        gr.Slider(
            minimum=4,
            maximum=6,
            step=1,
            value=4,
            label="Octave"
        ),
        gr.Number(
            value=1,
            label="Duration in seconds"
        ),
    ],
    outputs=gr.Audio(label="Generated Tone"),
    title="Generate Musical Tone",
    description="Choose a note, octave, and duration. The app will generate a sine wave tone."
)

print("Launching tone generation demo...")
tone_demo.launch(
    share=True,
    debug=True,
    show_error=True
)

Launching tone generation demo...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://09430d60e42cc6dcfe.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Generate tone function called.
Selected note index: 0
Selected note name: C
Selected octave: 4
Selected duration: 2
Calculated frequency: 261.63 Hz
Generated audio sample rate: 48000
Generated audio shape: (96000,)
Generated audio dtype: float32


/usr/local/lib/python3.12/dist-packages/gradio/processing_utils.py:688: UserWarning: Trying to convert audio automatically from float32 to 16-bit int format.
  warnings.warn(warning.format(data.dtype))


Created dataset file at: .gradio/flagged/dataset2.csv
Generate tone function called.
Selected note index: 9
Selected note name: A
Selected octave: 4
Selected duration: 2
Calculated frequency: 440.0 Hz
Generated audio sample rate: 48000
Generated audio shape: (96000,)
Generated audio dtype: float32


/usr/local/lib/python3.12/dist-packages/gradio/processing_utils.py:688: UserWarning: Trying to convert audio automatically from float32 to 16-bit int format.
  warnings.warn(warning.format(data.dtype))


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://09430d60e42cc6dcfe.gradio.live


In [9]:
!pip install -q soundfile librosa

In [10]:
print("=" * 80)
print("Loading automatic speech recognition model...")
print("Model: facebook/wav2vec2-base-960h")
print("This avoids the Whisper num_frames error.")
print("=" * 80)

asr_model = pipeline(
    task="automatic-speech-recognition",
    model="facebook/wav2vec2-base-960h",
    device=device
)

print("=" * 80)
print("ASR model loaded successfully.")
print("Using device:", "GPU" if device == 0 else "CPU")
print("=" * 80)

Loading automatic speech recognition model...
Model: facebook/wav2vec2-base-960h
This avoids the Whisper num_frames error.


config.json: 0.00B [00:00, ?B/s]

loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--facebook--wav2vec2-base-960h/snapshots/22aad52d435eb6dbaf354bdad9b0da84ce7d6156/config.json
Model config Wav2Vec2Config {
  "activation_dropout": 0.1,
  "adapter_attn_dim": null,
  "adapter_kernel_size": 3,
  "adapter_stride": 2,
  "add_adapter": false,
  "apply_spec_augment": true,
  "architectures": [
    "Wav2Vec2ForCTC"
  ],
  "attention_dropout": 0.1,
  "bos_token_id": 1,
  "classifier_proj_size": 256,
  "codevector_dim": 256,
  "contrastive_logits_temperature": 0.1,
  "conv_bias": false,
  "conv_dim": [
    512,
    512,
    512,
    512,
    512,
    512,
    512
  ],
  "conv_kernel": [
    10,
    3,
    3,
    3,
    3,
    2,
    2
  ],
  "conv_stride": [
    5,
    2,
    2,
    2,
    2,
    2,
    2
  ],
  "ctc_loss_reduction": "sum",
  "ctc_zero_infinity": false,
  "diversity_loss_weight": 0.1,
  "do_stable_layer_norm": false,
  "eos_token_id": 2,
  "feat_extract_activation": "gelu",

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

loading weights file model.safetensors from cache at /root/.cache/huggingface/hub/models--facebook--wav2vec2-base-960h/snapshots/22aad52d435eb6dbaf354bdad9b0da84ce7d6156/model.safetensors
Since the `dtype` attribute can't be found in model's config object, will use dtype={dtype} as derived from model's weights


...

vocab.json:   0%|          | 0.00/291 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

loading configuration file preprocessor_config.json from cache at /root/.cache/huggingface/hub/models--facebook--wav2vec2-base-960h/snapshots/22aad52d435eb6dbaf354bdad9b0da84ce7d6156/preprocessor_config.json
loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--facebook--wav2vec2-base-960h/snapshots/22aad52d435eb6dbaf354bdad9b0da84ce7d6156/config.json
Model config Wav2Vec2Config {
  "activation_dropout": 0.1,
  "adapter_attn_dim": null,
  "adapter_kernel_size": 3,
  "adapter_stride": 2,
  "add_adapter": false,
  "apply_spec_augment": true,
  "architectures": [
    "Wav2Vec2ForCTC"
  ],
  "attention_dropout": 0.1,
  "bos_token_id": 1,
  "classifier_proj_size": 256,
  "codevector_dim": 256,
  "contrastive_logits_temperature": 0.1,
  "conv_bias": false,
  "conv_dim": [
    512,
    512,
    512,
    512,
    512,
    512,
    512
  ],
  "conv_kernel": [
    10,
    3,
    3,
    3,
    3,
    2,
    2
  ],
  "conv_stride": [
    5,
    2,
    2,
    2,


ASR model loaded successfully.
Using device: GPU


In [11]:
def transcribe_audio(mic_audio=None, uploaded_audio=None):
    print("=" * 80)
    print("Transcribe audio function called.")
    print("Mic audio:", mic_audio)
    print("Uploaded audio:", uploaded_audio)

    if mic_audio is not None:
        audio_path = mic_audio
        print("Using microphone audio.")
    elif uploaded_audio is not None:
        audio_path = uploaded_audio
        print("Using uploaded audio file.")
    else:
        print("No audio provided.")
        print("=" * 80)
        return "You must either provide a mic recording or upload an audio file."

    print("Audio path:", audio_path)
    print("Running ASR...")
    start_time = time.time()

    try:
        result = asr_model(audio_path)
        transcription = result["text"]

        end_time = time.time()

        print("Transcription:")
        print(transcription)
        print("Processing time:", round(end_time - start_time, 2), "seconds")
        print("=" * 80)

        return transcription

    except Exception as e:
        print("ASR failed.")
        print("Error:", repr(e))
        print("=" * 80)
        return f"ASR failed with error: {repr(e)}"


asr_demo = gr.Interface(
    fn=transcribe_audio,
    inputs=[
        gr.Audio(
            sources=["microphone"],
            type="filepath",
            label="Record Audio"
        ),
        gr.Audio(
            sources=["upload"],
            type="filepath",
            label="Upload Audio File"
        ),
    ],
    outputs=gr.Textbox(
        lines=5,
        label="Transcription"
    ),
    title="Speech Recognition",
    description="Record audio or upload an audio file. This version uses Wav2Vec2 instead of Whisper."
)

print("Launching speech recognition demo...")
asr_demo.launch(
    share=True,
    debug=True,
    show_error=True
)

Launching speech recognition demo...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://84bb69e77254107f8f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 421, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 56, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1159, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error

Transcribe audio function called.
Mic audio: None
Uploaded audio: /tmp/gradio/7f51e000b92e59127cffd858165ec60d964c918388c9fb87e0ec380298362d42/freesound_community-54321-35890.mp3
Using uploaded audio file.
Audio path: /tmp/gradio/7f51e000b92e59127cffd858165ec60d964c918388c9fb87e0ec380298362d42/freesound_community-54321-35890.mp3
Running ASR...
Transcription:
TEN NINE EIGHT SEVEN SIX FIVE FOUR THREE TWO ONE
Processing time: 1.28 seconds
Created dataset file at: .gradio/flagged/dataset3.csv
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://84bb69e77254107f8f.gradio.live
